# Multi-agentes de IA para análises de revisões de papers

  - Cria vector store (FAISS) a partir de embbedings calculados com modelo disponibilizado na Hugging Face.

  - Realiza buscas por similaridade.
  
  - Usa prompts com retrievers.

In [1]:
import warnings
warnings.filterwarnings('ignore')

import json
import re
from rich import print
from tqdm import tqdm
from typing import List
import pickle

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS

from sklearn.decomposition import PCA
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)

DATASET_PATH = 'datasets/reviews.json'
EXPORT_TABLES = True
EXPORT_EMBEDDINGS = True

MAX_ROWS = 10000  # limita número de linhas para testes

In [2]:
with open(DATASET_PATH, 'r', encoding='utf-8') as f:
    dataset = json.load(f)


## Prepara dataset

### Normaliza JSON de entrada

In [3]:
df_reviews = pd.json_normalize(
    dataset,
    record_path=['paper', 'review'],
    meta=[['paper', 'id']],
)

df_reviews.rename(columns={'paper.id': 'paper_id'}, inplace=True)
print(df_reviews.shape)


df_reviews.text = df_reviews.text.apply(lambda x: pd.NA if len(x)==0 else x)
df_reviews.dropna(inplace=True)

print(df_reviews.shape)
df_reviews.head(1)

(405, 9)

(397, 9)

,confidence,evaluation,id,lan,orientation,remarks,text,timespan,paper_id
0,4,1,1,es,0,,"- El artículo aborda un problema contingente y muy relevante, e incluye tanto un diagnóstico nacional de uso de buenas prácticas como una solución (buenas prácticas concretas). - El lenguaje es adecuado. - El artículo se siente como la concatenación de tres artículos diferentes: (1) resultados de una encuesta, (2) buenas prácticas de seguridad, (3) incorporación de buenas prácticas. - El orden de las secciones sería mejor si refleja este orden (la versión revisada es #2, #1, #3). - El artículo no tiene validación de ningún tipo, ni siquiera por evaluación de expertos.",2010-07-05,1


### Utils

In [4]:
def export_as_parquet(table_name: str, pandas_df: pd.DataFrame) -> None:
    table_path = f'datasets/{table_name}.parquet'
    pandas_df.to_parquet(table_path, index=False)
    print(f'Criado arquivo `{table_path}`')
    print(f'Dimensões: {pandas_df.shape}')

### Exporta como parquet

In [5]:
EXPORT_TABLES = True
if EXPORT_TABLES:
    export_as_parquet(table_name='paper_reviews', pandas_df=df_reviews)

Criado arquivo `datasets/paper_reviews.parquet`

Dimensões: (397, 9)

## Chunking

In [6]:
CHUNK_SIZE = 70
OVERLAP = 10

def chunk_text(text, chunk_size=200, overlap=50):
    chunks = []
    start = 0
    text_length = len(text)

    while start < text_length:
        end = start + chunk_size
        chunks.append(text[start:end])
        start = end - overlap

    return chunks

In [7]:
rows = []
for _, row in df_reviews.iterrows():
    chunks = chunk_text(row['text'], chunk_size=CHUNK_SIZE, overlap=OVERLAP)
    for i, chunk in enumerate(chunks):
        rows.append({
            'review_uri': f"paper_{row['paper_id']}_review_{row['id']}_chunk_{i}",
            'paper_id': row['paper_id'],
            'review_id': row['id'],
            'chunked_text': chunk,
            'chunk_size': len(chunk),
            'timespan': row['timespan']
        })

df_chunks = pd.DataFrame(rows)
print(df_chunks.shape)
df_chunks.head()

(6932, 6)

,review_uri,paper_id,review_id,chunked_text,chunk_size,timespan
0,paper_1_review_1_chunk_0,1,1,"- El artículo aborda un problema contingente y muy relevante, e incluy",70,2010-07-05
1,paper_1_review_1_chunk_1,1,1,", e incluye tanto un diagnóstico nacional de uso de buenas prácticas c",70,2010-07-05
2,paper_1_review_1_chunk_2,1,1,rácticas como una solución (buenas prácticas concretas). - El lenguaje,70,2010-07-05
3,paper_1_review_1_chunk_3,1,1,l lenguaje es adecuado. - El artículo se siente como la concatenación,70,2010-07-05
4,paper_1_review_1_chunk_4,1,1,catenación de tres artículos diferentes: (1) resultados de una encuest,70,2010-07-05


In [8]:
df_chunks.chunk_size.value_counts(dropna=False)

chunk_size
70    6474
39      15
34      14
25      12
35      11
45      11
14      11
49      10
19      10
62      10
2       10
30       9
12       9
69       9
9        9
60       9
17       9
31       9
68       8
8        8
32       8
27       8
33       8
47       8
64       8
4        8
67       8
7        8
43       8
51       8
29       8
42       7
36       7
21       7
57       7
38       6
20       6
48       6
61       6
1        6
41       6
40       6
18       5
22       5
10       5
65       5
5        5
44       5
54       5
52       5
56       5
13       5
59       4
58       4
50       4
66       4
6        4
23       4
55       4
15       3
46       3
26       3
11       3
63       3
3        3
53       3
28       3
24       3
16       2
Name: count, dtype: int64

In [9]:
df_chunks = df_chunks[df_chunks['chunk_size'] == CHUNK_SIZE]
print(df_chunks.shape)
df_chunks.head()

(6474, 6)

,review_uri,paper_id,review_id,chunked_text,chunk_size,timespan
0,paper_1_review_1_chunk_0,1,1,"- El artículo aborda un problema contingente y muy relevante, e incluy",70,2010-07-05
1,paper_1_review_1_chunk_1,1,1,", e incluye tanto un diagnóstico nacional de uso de buenas prácticas c",70,2010-07-05
2,paper_1_review_1_chunk_2,1,1,rácticas como una solución (buenas prácticas concretas). - El lenguaje,70,2010-07-05
3,paper_1_review_1_chunk_3,1,1,l lenguaje es adecuado. - El artículo se siente como la concatenación,70,2010-07-05
4,paper_1_review_1_chunk_4,1,1,catenación de tres artículos diferentes: (1) resultados de una encuest,70,2010-07-05


In [10]:
print(f'df_reviews.shape: {df_reviews.shape}')
print(f'df_chunks.shape: {df_chunks.shape}')

df_reviews.shape: (397, 9)

df_chunks.shape: (6474, 6)

In [11]:
if MAX_ROWS:
    df_chunks = df_chunks[:MAX_ROWS]

df_chunks.shape

(6474, 6)

## Gera embeddings

In [12]:
embed_model = HuggingFaceEmbeddings(
    model_name='BAAI/bge-base-en-v1.5',
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [13]:
def normalize_text(text: str) -> List[float]:
    if text is None:
        return None
    
    t_norm = str(text).strip()
    t_norm = re.sub(r'\s+', ' ', t_norm) 
    t_norm = t_norm.replace('\n', ' ').replace('\r', ' ')
    
    return t_norm


def get_embeddings(text: List[str]) -> List[float]:
    if not text:
        return []
    
    text_norm = normalize_text(text)
    embed_text = embed_model.embed_query(text_norm)

    return embed_text

In [14]:
low_dim = 100

pca = PCA(
    n_components=min(df_chunks.shape[0], low_dim),
    random_state=42
)
pca

,"n_components n_components: int, float or 'mle', default=NoneNumber of components to keep.if n_components is not set all components are kept:: n_components == min(n_samples, n_features)If ``n_components == 'mle'`` and ``svd_solver == 'full'``, Minka'sMLE is used to guess the dimension. Use of ``n_components == 'mle'``will interpret ``svd_solver == 'auto'`` as ``svd_solver == 'full'``.If ``0 < n_components < 1`` and ``svd_solver == 'full'``, select thenumber of components such that the amount of variance that needs to beexplained is greater than the percentage specified by n_components.If ``svd_solver == 'arpack'``, the number of components must bestrictly less than the minimum of n_features and n_samples.Hence, the None case results in:: n_components == min(n_samples, n_features) - 1",100
,"copy copy: bool, default=TrueIf False, data passed to fit are overwritten and runningfit(X).transform(X) will not yield the expected results,use fit_transform(X) instead.",True
,"whiten whiten: bool, default=FalseWhen True (False by default) the `components_` vectors are multipliedby the square root of n_samples and then divided by the singular valuesto ensure uncorrelated outputs with unit component-wise variances.Whitening will remove some information from the transformed signal(the relative variance scales of the components) but can sometimeimprove the predictive accuracy of the downstream estimators bymaking their data respect some hard-wired assumptions.",False
,"svd_solver svd_solver: {'auto', 'full', 'covariance_eigh', 'arpack', 'randomized'}, default='auto'""auto"" : The solver is selected by a default 'auto' policy is based on `X.shape` and `n_components`: if the input data has fewer than 1000 features and more than 10 times as many samples, then the ""covariance_eigh"" solver is used. Otherwise, if the input data is larger than 500x500 and the number of components to extract is lower than 80% of the smallest dimension of the data, then the more efficient ""randomized"" method is selected. Otherwise the exact ""full"" SVD is computed and optionally truncated afterwards.""full"" : Run exact full SVD calling the standard LAPACK solver via `scipy.linalg.svd` and select the components by postprocessing""covariance_eigh"" : Precompute the covariance matrix (on centered data), run a classical eigenvalue decomposition on the covariance matrix typically using LAPACK and select the components by postprocessing. This solver is very efficient for n_samples >> n_features and small n_features. It is, however, not tractable otherwise for large n_features (large memory footprint required to materialize the covariance matrix). Also note that compared to the ""full"" solver, this solver effectively doubles the condition number and is therefore less numerical stable (e.g. on input data with a large range of singular values).""arpack"" : Run SVD truncated to `n_components` calling ARPACK solver via `scipy.sparse.linalg.svds`. It requires strictly `0 < n_components < min(X.shape)`""randomized"" : Run randomized SVD by the method of Halko et al... versionadded:: 0.18.0.. versionchanged:: 1.5 Added the 'covariance_eigh' solver.",'auto'
,"tol tol: float, default=0.0Tolerance for singular values computed by svd_solver == 'arpack'.Must be of range [0.0, infinity)... versionadded:: 0.18.0",0.0
,"iterated_power iterated_power: int or 'auto', default='auto'Number of iterations for the power method computed bysvd_solver == 'randomized'.Must be of range [0, infinity)... versionadded:: 0.18.0",'auto'
,"n_oversamples n_oversamples: int, default=10This parameter is only relevant when `svd_solver=""randomized""`.It corresponds to the additional number of random vectors to sample therange of `X` so as to ensure proper conditioning. See:func:`~sklearn.utils.extmath.randomized_svd` for more details... versionadded:: 1.1",10
,"power_iteration_normalizer power_iteration_normalizer: {'auto', 'QR', 'LU', 'none'}, default='auto'Power iteration normalizer for randomized SV

In [15]:
texts = df_chunks['chunked_text'].tolist()
print(f'# amostras: {len(texts)} ')

all_embeddings = []
for i in tqdm(range(0, len(texts)), desc='Processing embeddings'):
    batch = texts[i]
    emb_high_dim = get_embeddings(batch)
    all_embeddings.append(emb_high_dim)

# amostras: 6474

Processing embeddings: 100%|██████████| 6474/6474 [01:51<00:00, 58.10it/s]


In [16]:
x_pca = np.array(all_embeddings)
print(f'Dimensões antes do PCA: {len(all_embeddings[0])}')
all_embeddings_low_dim = pca.fit_transform(x_pca)
print(f'Dimensões após redução de dimensionalidade: {len(all_embeddings_low_dim[0])}')

Dimensões antes do PCA: 768

Dimensões após redução de dimensionalidade: 100

In [17]:
df_chunks['embedding'] = all_embeddings_low_dim.tolist()
print(df_chunks.shape)
df_chunks.head(1)

(6474, 7)

,review_uri,paper_id,review_id,chunked_text,chunk_size,timespan,embedding
0,paper_1_review_1_chunk_0,1,1,"- El artículo aborda un problema contingente y muy relevante, e incluy",70,2010-07-05,"[-0.04430300143039605, 0.006216770554445256, 0.12915026418651745, 0.06797646070248231, -0.024447868346449415, -0.11695544777802395, 0.11994276304275446, -0.12888593240384932, 0.07694521895033483, -0.02552701042441331, 0.035130253674070605, -0.021290023352751417, -0.039546974057021164, -0.02088823660471014, -0.014113930477334587, 0.07748321733681499, -0.014336564710082118, 0.08839047044865048, -0.21703177752108263, 0.09036225885932778, 0.04267659448412163, -0.034168997805158266, 0.047874076655974825, 0.002284670547358691, 0.023585547012217392, -0.024919069886372547, -0.056788747631129226, -0.008345370225731985, 0.002701782288786598, 0.030729398897098647, -0.086951326661941, 0.04755698886049043, -0.0078965457714073, -0.024383098247516662, 0.003648534933416896, -0.018378855672581226, 0.00973375548305387, 0.008424006549231634, -0.002051591844123525, -0.012948956320072237, 0.01137523257293282, 0.026418185872515747, -0.002649577486628258, -0.011903636668665871, 0.019072396054665496, -0.060436725032566016, 0.009426979249604235, 0.011670349109803396, -0.022168784708812454, 0.015344946570562774, -0.009769237995414539, -0.01335291727710848, -0.024670585206046085, 0.016301974982089604, -0.03470558871632932, -0.013746374514095333, 0.01587317152879185, 0.0023681986322640442, 0.03498559411580699, -0.010842520417089937, 0.009851888118778043, 0.0009168384089338133, 0.029896896864254976, -0.004896437897360193, 0.041647311556600425, -0.05569271401154171, 0.006358394815410071, 0.023165730278650112, -0.06345769795520016, -0.04045463416243082, 0.00940572122602729, -0.06302393092492557, -0.05631815632744301, 0.012635480432720655, 0.0022147608612476874, 0.03288285634626698, 0.02114540387051425, 0.004458266623712661, -0.04629460213283339, -0.005923875669082454, 0.011116219665925669, -0.013755740134064652, -0.0058250917791114465, 0.041679064696009314, 0.003229365699237482, -0.01511040130096131, 0.016036089795722765, -0.00432261879809009, 0.03255018679342226, -0.02849385353931009, 0.003773494353697112, -0.007047849480189186, 0.004418004124004104, -0.06709285867887897, 0.05049259142176857, 0.00822200888237321, -0.00418454587003588, -0.031885825065119895, 0.004414750199753011, 0.011213136023458133]"


In [18]:
with open('models/pca_model.pkl', 'wb') as f:
    pickle.dump(pca, f)

In [19]:
if EXPORT_EMBEDDINGS:
  export_as_parquet(table_name='paper_reviews_embeddings', pandas_df=df_chunks)

Criado arquivo `datasets/paper_reviews_embeddings.parquet`

Dimensões: (6474, 7)

## Cria vector store (FAISS)

In [20]:
text_embeddings = np.vstack([np.array(e, dtype=np.float32) for e in df_chunks['embedding']])

In [21]:
text_embedding_pairs = zip(texts, text_embeddings)

In [22]:
vs_faiss = FAISS.from_embeddings(text_embedding_pairs, embed_model)

### Exporta vector store

In [23]:
vs_faiss.save_local(f'datasets/vectorstore.faiss')

### Testa busca semântica

In [24]:
sentence_search = 'prácticas de seguridad en desarrollo de software'
sentence_search_emb_low_dim = np.array([get_embeddings(sentence_search)])
emb_vector_sent_search = pca.transform(sentence_search_emb_low_dim)

In [25]:
emb_vector_sent_search[0]

array([ 0.21047162,  0.22966825, -0.10830101,  0.0023296 ,  0.08895827,
        0.07075647,  0.00864984, -0.06551756,  0.13522349,  0.0334992 ,
       -0.02616684, -0.057484  ,  0.00501122,  0.01198524, -0.07948197,
        0.11322978,  0.21874498,  0.0190027 ,  0.07158601,  0.02996279,
        0.02899561,  0.01343608,  0.01895387,  0.05877096, -0.02542484,
        0.13129122,  0.08421585, -0.09257601, -0.08130073, -0.00978167,
        0.00416404, -0.05849631,  0.07651415, -0.04784206, -0.03654826,
        0.07049733,  0.06855678,  0.04959529,  0.051506  , -0.01489743,
        0.04262009, -0.05032459,  0.07044796, -0.03438276,  0.02225554,
       -0.01572126, -0.03072109, -0.00652414, -0.05543822,  0.00233008,
       -0.04327633, -0.06707498,  0.0193779 ,  0.00804188, -0.02063438,
       -0.05438561,  0.06124749,  0.03309815, -0.02853281, -0.04556211,
       -0.04108091,  0.00424927, -0.02439494,  0.02055979, -0.03165642,
        0.00642043, -0.01450524, -0.02296474,  0.07506925, -0.02

In [26]:
vs_faiss.similarity_search_with_score_by_vector(emb_vector_sent_search[0], k=3)

[(Document(id='028ba21a-f1e7-446b-8163-67a531c55afd', metadata={}, page_content='el desarrollo de software seguro, pero la guía presenta 10 prácticas. '),
  np.float32(0.07574441)),
 (Document(id='317731ca-ae50-40bd-b501-fe54191af8b2', metadata={}, page_content='rollo de software seguro. Se describen las mejores prácticas recomenda'),
  np.float32(0.0889175)),
 (Document(id='9c91f5c0-628b-442c-86d8-9d23e671f0e8', metadata={}, page_content='El trabajo presenta una revisión general de la seguridad de software e'),
  np.float32(0.095238894))]